# Q-capture backpressure metrics benchmark

This notebook runs the checked-in Q-ring capture benchmark and asserts the ownership and metric invariants used by this PR. It defaults to CUDA and is bounded to the GPU selected with `CUDA_VISIBLE_DEVICES`; set `Q_CAPTURE_DEVICE=cpu` for a functional-only run. Run all cells from the LMCache repository root.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Standard
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

repo_candidates = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in repo_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("run from the LMCache repository")
device = os.environ.get("Q_CAPTURE_DEVICE", "cuda")
script = repo_root / "benchmarks/microbenchmark/q_capture_observability_benchmark.py"
command = [
    sys.executable,
    str(script),
    "--device",
    device,
    "--requests",
    "8",
    "--tokens-per-request",
    "256",
    "--block-size",
    "16",
    "--hidden-dim",
    "128",
    "--warmup",
    "10",
    "--iterations",
    "101",
    "--snapshot-iterations",
    "10000",
]
print("$", shlex.join(command))
completed = subprocess.run(
    command, check=True, capture_output=True, text=True, cwd=repo_root
)
print(completed.stdout)
evidence = json.loads(completed.stdout)
assert evidence["invariants"]["ring_fully_reclaimed"] is True
assert evidence["invariants"]["sample_count"] is True
metrics = evidence["capture_metrics"]
assert metrics["steps_attempted"] == 111
assert metrics["steps_captured"] == 111
assert metrics["steps_skipped"] == 0
assert metrics["ring"]["used_blocks"] == 0
assert metrics["ring"]["high_watermark_blocks"] == 128
print("Q-capture metric and ring-ownership checks passed")